# Exploratory Data Analysis

Examine the full merged table before cleaning, including every observed order status. The observations below refer to the reference merge of 112,650 items.

**Contents**

1. [Project setup and input](#eda-setup) — Load the merged table and identify the input version.
2. [Table structure and completeness](#eda-structure) — Establish the unit of observation for each profile.
3. [Distributions](#eda-distributions) — Examine prices, measurements, categories and timestamps.
4. [Missing data](#eda-missingness) — Locate nulls and identify fields that are missing together.
5. [Domain checks](#eda-domain) — Check identifiers, event sequences, measurements and order structure.
6. [Record-level investigations](#eda-records) — Trace flagged patterns to individual records.
7. [Input preservation](#eda-preservation) — Check the original item table after profiling.

Sections 3 and 4 use the entity tables from section 2. Section 5 also uses outputs from sections 3 and 4. Section 6 follows up on the profiles in sections 2–5.

[User Guide](../docs/EDA/User_Guide.md) · [Technical Reference](../docs/EDA/Technical_Reference.md) · [EDA Report](../reports/EDA/EDA_Report.md)


<a id="eda-setup" name="eda-setup"></a>

## 1. Project setup and input

Load the interim dataset and record its initial state for the final input check.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


# Locate the repository before importing project modules.
def find_project_root(start_path: Path) -> Path:
    """Find the project root containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Could not locate project root containing both 'src/' and 'requirements.txt'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.data_loader import load_data
from src.eda.checks import build_entity_views, profile_entity_consistency, profile_keys
from src.eda.profiling import get_column_roles, profile_columns, profile_dataset
from src.eda.workflow import (
    inspect_numeric_distributions,
    inspect_category_distributions,
    inspect_temporal_distributions,
    inspect_missing_patterns,
    inspect_missing_pairs,
    inspect_conditional_missingness,
    inspect_status_missingness,
    inspect_text_formats,
    inspect_order_timeline,
    inspect_measurement_plausibility,
    inspect_order_structure,
    inspect_timeline_details,
    inspect_delivered_completeness,
    inspect_product_completeness,
    inspect_numeric_tails,
    inspect_deadline_calendar,
    inspect_seller_city_spacing,
    inspect_structure_overlap,
)
from src.paths import INTERIM_DIR
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim data path: {INTERIM_DIR}")


### Load the full merged table


In [ ]:
df = load_data("merged_order_items", stage="interim")

# Record the loaded input without making a second copy of the full dataset.
input_baseline = {
    "shape": df.shape,
    "columns": df.columns.copy(),
    "dtypes": df.dtypes.copy(),
    "row_hashes": pd.util.hash_pandas_object(df, index=True).copy(),
}


### Reference dimensions

Compare item, column and order counts with the reference merge.


In [ ]:
baseline_summary = pd.DataFrame(
    [
        ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
        ["Unique orders", 98_666, df["order_id"].nunique()],
    ],
    columns=["Metric", "Reference", "Observed"],
)
baseline_summary["Matches reference"] = baseline_summary["Reference"].eq(
    baseline_summary["Observed"]
)
display(baseline_summary)

if not baseline_summary["Matches reference"].all():
    print("Reference counts differ. Check the input version before interpreting later profiles.")

print("Input baseline retained in memory for the final non-mutation check.")


The observed dimensions match the reference merge across all three counts.


### Order-item preview

Read the first five item records with their order, seller and product fields side by side.


In [ ]:
df.head()

Each preview row brings the item's price and freight together with order timestamps. An order with several items will contribute several rows to a full-table count.


<a id="eda-structure" name="eda-structure"></a>

## 2. Table structure and completeness

Start with item rows, then check whether repeated order, product, customer and seller attributes agree before counting distinct entities.

### Dataset dimensions

Count item rows and distinct entities. Each `customer_id` identifies a customer record associated with a purchase.


In [ ]:
dataset_profile = profile_dataset(df)
display(dataset_profile)


There are no exact duplicate rows. The 112,650 item rows represent 98,666 orders.


### Column roles and completeness

Percentages use all item rows. Blank strings and nulls are counted separately.


In [ ]:
column_roles = get_column_roles()
column_profile = profile_columns(df)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Registered column roles")
    display(column_roles)
    print("Completeness at item grain")
    display(column_profile)


Nulls occur in eight columns, all related to order events or product attributes. Price, freight and the identifier columns are complete.


### Keys and repeated identifiers

Items are unique by `(order_id, order_item_id)`. An order ID can repeat across items.


In [ ]:
key_profile = profile_keys(df)
with pd.option_context("display.max_columns", None):
    display(key_profile)


The item key is unique and complete. Repeated order IDs are expected because an order can contain several items.


### Attribute consistency within each ID

A mix of missing and observed values for the same attribute counts as a conflict.


In [ ]:
entity_consistency = profile_entity_consistency(df)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Consistency by entity")
    display(entity_consistency["summary"])
    print("Consistency by attribute")
    display(entity_consistency["attributes"])

if entity_consistency["conflicts"].empty:
    print("No conflicting entity attributes found.")
else:
    print("Conflicting entity attributes (all detected entity/attribute pairs)")
    display(entity_consistency["conflicts"])


Repeated records agree on the checked attributes within every order, customer, product and seller ID. A single row per entity can therefore represent those attributes.


### One row per source key

Build separate views for orders, customer records, products and sellers using their source keys.


In [ ]:
entity_views = build_entity_views(df)
entity_view_summary = pd.DataFrame(
    [
        [name, len(view), len(view.columns)]
        for name, view in entity_views.items()
    ],
    columns=["View", "Observed rows", "Columns"],
)
display(entity_view_summary)


The product view contains 32,951 products and the seller view 3,095 sellers. These are the denominators for their respective profiles.


<a id="eda-distributions" name="eda-distributions"></a>

## 3. Distributions

Use each field's source unit so that products and orders with several items receive the appropriate weight.

### Prices, freight and product measurements

Price and freight use item rows. Weight and dimensions use distinct products. The zoomed histograms show the 1st to 99th percentiles.


In [ ]:
numeric_profiles = inspect_numeric_distributions(
    entity_views=entity_views,
    column_roles=column_roles,
)


Item prices have a long upper tail. The median is BRL 74.99, the 99th percentile is BRL 890 and the maximum is BRL 6,735.


### Categories and ZIP prefixes

Frequencies count distinct orders for status, products for category, and customer or seller records for location.


In [ ]:
category_distributions = inspect_category_distributions(
    entity_views=entity_views,
    column_roles=column_roles,
)


Delivered orders account for 96,478 of 98,666 orders. SP accounts for about 42% of customer records and 60% of sellers, so the geographic distributions differ on the two sides of an order.


### Timestamp coverage by month, date, weekday and hour

Shipping deadlines are counted per item. Order events are counted per order using the recorded clock.


In [ ]:
temporal_distributions = inspect_temporal_distributions(
    entity_views=entity_views,
    column_roles=column_roles,
)


Purchases peak at 16:00 and approvals at 02:00 on the recorded clock. Every estimated delivery timestamp is at midnight, so its date carries the useful variation.


<a id="eda-missingness" name="eda-missingness"></a>

## 4. Missing data

Compare missing fields first within each source table view and then across status, category and location groups.

### Patterns of missing fields

Each rate uses the rows in its own view. An empty pattern `()` means all selected fields are present.


In [ ]:
missing_patterns = inspect_missing_patterns(
    df=df,
    entity_views=entity_views,
    column_profile=column_profile,
)


At least one field is missing on 4,007 item rows. At order level, the most common missing core-event pattern is customer delivery alone (1,182 orders).


### Missing values in pairs of columns

Read the heatmap across a row. Among records missing the row field, each cell gives the percentage also missing the column field.


In [ ]:
missing_pair_profiles = inspect_missing_pairs(
    missingness_frames=missing_patterns["missingness_frames"],
    native_column_profiles=missing_patterns["native_column_profiles"],
)


Of 1,009 orders missing carrier handoff, 1,008 also lack customer delivery. In the reverse direction the share is 46.0%, because many more orders lack customer delivery.


### Missing values by status, purchase month, category and location

Within each group, compare the number affected with the number observed.


In [ ]:
missing_context = inspect_conditional_missingness(
    column_roles=column_roles,
    df=df,
    entity_views=entity_views,
)


Missing physical measurements affect two distinct products but 18 item rows. Product counts describe the catalog gap, while item counts show how often it appears in orders.


### Missing timestamps by order status

Each percentage uses all orders with that status.


In [ ]:
status_missing_rates = inspect_status_missingness(
    condition_profiles=missing_context["condition_profiles"],
    order_timestamp_columns=missing_context["order_timestamp_columns"],
)


All 1,106 shipped orders lack at least one core timestamp. Among 96,478 delivered orders, only 23 do. Timestamp completeness varies sharply by status.


<a id="eda-domain" name="eda-domain"></a>

## 5. Domain checks

Test whether observed values follow their expected formats, event order and physical ranges.

### Text, identifiers and location codes

Inspect the recorded spellings and compare IDs, ZIP prefixes and state codes with their expected patterns.


In [ ]:
text_format_profiles = inspect_text_formats(
    column_roles=column_roles,
    missingness_frames=missing_patterns["missingness_frames"],
)


The checked IDs, ZIP prefixes and state codes match their expected formats. Three seller records contain repeated spaces inside a city name.


### Order of recorded events

Each rate uses orders with both timestamps. Delivery estimates are compared with purchase dates at calendar-day precision.


In [ ]:
order_timeline = inspect_order_timeline(
    entity_views=entity_views,
)


The most frequent reversal is handoff before approval (1,359 orders). Another 23 orders record customer delivery before carrier handoff.


### Unusual measurements and item deadlines

Inspect zero, negative and missing measurements, then compare each shipping deadline with the purchase time.


In [ ]:
measurement_checks = inspect_measurement_plausibility(
    numeric_profiles=numeric_profiles,
    df=df,
)


Freight is zero on 383 items and weight is zero for four products. No item has a shipping deadline before purchase.


### Items, sellers and deadlines per order

Count distinct products, sellers and shipping deadlines within each order.


In [ ]:
order_structure = inspect_order_structure(
    df=df,
)


Of 9,803 orders containing multiple items, 1,278 involve multiple sellers. A single seller field cannot describe every order.


<a id="eda-records" name="eda-records"></a>

## 6. Record-level investigations

Examine the records behind the flags to distinguish repeated exposure to the same entity from separate cases.

### Reversed timestamps

Separate reversals within a calendar day from those crossing a date boundary. The plots count orders within each flagged group.


In [ ]:
timeline_details = inspect_timeline_details(
    entity_views=entity_views,
    timeline_profile=order_timeline["timeline_profile"],
    order_context=missing_context["order_context"],
)


Of the 1,359 handoffs recorded before approval, 679 fall on the same date and 680 on an earlier date. A date-only check would miss nearly half of these reversals.


### Missing purchase, approval, handoff or delivery times in delivered orders

Check which core events are missing or out of sequence among orders marked as delivered.


In [ ]:
delivered_completeness = inspect_delivered_completeness(
    orders=entity_views["orders"],
    core_timestamp_columns=missing_context["core_timestamp_columns"],
    any_ordering_flag=timeline_details["any_ordering_flag"],
    order_context=missing_context["order_context"],
)


Among delivered orders, 23 lack a core timestamp and a separate 1,373 have complete timestamps with an ordering flag. Completeness alone would miss the latter group.


### Missing or nonpositive product measurements

A single affected product can appear in several items and orders.


In [ ]:
product_completeness = inspect_product_completeness(
    entity_views=entity_views,
    product_measurement_columns=missing_context["product_measurement_columns"],
    df=df,
    orders=entity_views["orders"],
)


The two products missing measurements lack all four physical fields. Four different products have a recorded weight of zero, affecting eight items across six orders.


### Extreme measurements by product category

The category breakdown uses values outside the 1.5-IQR fences for each measurement.


In [ ]:
numeric_tails = inspect_numeric_tails(
    numeric_profiles=numeric_profiles,
    df=df,
    entity_views=entity_views,
)


The highest observed item price (BRL 6,735) occurs in `utilidades_domesticas`, while the highest freight charge (BRL 409.68) occurs in `bebes`. The two extremes belong to different product categories.


### Shipping deadlines beyond the last purchase year

Inspect the order, seller and purchase date for each deadline beyond the last observed purchase year.


In [ ]:
deadline_calendar = inspect_deadline_calendar(
    orders=entity_views["orders"],
    df=df,
    order_timestamp_columns=missing_context["order_timestamp_columns"],
)


Four items from one seller have deadlines in 2020. They belong to three orders purchased in 2017, including one delivered order.


### Repeated spaces in seller city names

`[space]` marks each space in the displayed examples.


In [ ]:
seller_city_spacing = inspect_seller_city_spacing(
    entity_views=entity_views,
    df=df,
    orders=entity_views["orders"],
)


The three affected seller cities also appear with single spaces in the same states. Keeping both spellings would split otherwise matching city labels.


### Orders with multiple sellers, deadlines or state pairs

Cross-tabulate multi-seller orders by the number of deadlines and the mix of customer and seller states.


In [ ]:
structure_overlap = inspect_structure_overlap(
    order_composition=order_structure["order_composition"],
    orders=entity_views["orders"],
    df=df,
)


All 336 orders with multiple deadlines have multiple sellers. Within each order–seller pair, the recorded deadline is consistent.


<a id="eda-preservation" name="eda-preservation"></a>

## 7. Input preservation

Compare the final item table with the baseline captured at load time, including column types, indexed values and row order.


In [ ]:
assert df.shape == input_baseline["shape"], "Input shape changed during profiling."
pd.testing.assert_index_equal(df.columns, input_baseline["columns"])
pd.testing.assert_series_equal(df.dtypes, input_baseline["dtypes"])
pd.testing.assert_series_equal(
    pd.util.hash_pandas_object(df, index=True), input_baseline["row_hashes"],
)
print("Input preserved after complete pre-cleaning EDA.")

The item table matches its starting state after profiling.
